In [13]:
import pandas as pd
import sys
import os
# Add parent directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.db_connection import DatabaseConnection
from datetime import datetime
from sqlalchemy import text
import numpy as np

In [ ]:

def get_max_transformed_date(db_conn: DatabaseConnection) -> datetime.date:
    """
        Function to return the max date of the column from the bronze.raw_sales
        to be used in fetching the new inserted data into bronze
    """

    with db_conn.engine.begin() as query_conn:
        result = query_conn.execute(text(""" SELECT max(transformed_date) FROM SILVER.cleansed_sales; """))
    return result.scalar().date()



def read_data_from_silver(db_conn: DatabaseConnection) ->pd.DataFrame:

    return db_conn.read_dataframe_from_db(f"SELECT * FROM Silver.cleansed_sales where transformed_date >= DATE '{get_max_transformed_date(db_conn)}'")



def fill_date_dim_table(cleansed_data: pd.DataFrame, db_connection: DatabaseConnection) -> pd.DataFrame:
    """
    Fill date dimension table for the ingestion date.

    Returns:
        DataFrame with single date dimension record
    """
    

    date_dim = cleansed_data[['invoice_date']].copy()

    date_dim['date_key'] = date_dim['invoice_date'].apply(lambda x: int(x.strftime("%Y%m%d")))
    date_dim['full_date'] = date_dim['invoice_date'].apply(lambda x: x.date())
    date_dim['day_of_week'] = date_dim['invoice_date'].apply(lambda x: x.date().weekday()+ 1)
    date_dim['day_of_month'] = date_dim['invoice_date'].apply(lambda x: x.date().day)
    date_dim['day_name'] = date_dim ['invoice_date'].apply(lambda x: x.date().strftime("%A"))
    date_dim['week_of_year'] = date_dim['invoice_date'].apply(lambda x: x.date().isocalendar()[1])
    date_dim['month'] = date_dim['invoice_date'].apply(lambda x: x.date().month)
    date_dim['month_name']=date_dim['invoice_date'].apply(lambda x: x.date().strftime("%B"))
    date_dim['quarter']=date_dim['invoice_date'].apply(lambda x: (x.date().month - 1 )// 3 + 1)
    date_dim['year'] = date_dim['invoice_date'].apply(lambda x: x.date().year)
    date_dim['is_weekend'] = date_dim['invoice_date'].apply(lambda x: x.date().weekday() >= 5)


    all_stored_dates = db_connection.read_dataframe_from_db("SELECT * FROM GOLD.DATE_DIM;")

    date_dim = date_dim.merge(all_stored_dates , how='left' , on=['date_key'] , indicator=True, suffixes=("","X"))


    date_dim = date_dim[date_dim['_merge'] == 'left_only']
    

    date_dim = date_dim[["date_key","full_date","day_of_week","day_of_month","day_name","week_of_year","month","month_name","quarter","year","is_weekend"]]
    print(date_dim.head())
    date_dim = date_dim.drop_duplicates(subset=['date_key'])
    # 0 if len(date_dim) == 0 else db_connection.load_dataframe_into_db(date_dim , 'gold' , 'date_dim')    
    return  date_dim



def fill_customer_dim_table(cleansed_data: pd.DataFrame , db_conn:DatabaseConnection) -> pd.DataFrame:

    customer_data = cleansed_data.copy()
    
    known_customers = customer_data[ customer_data['customer_type'] =='registered']
    unknown_customers = customer_data[ customer_data['customer_type'] =='guest']
    
    all_existing_customers = db_conn.read_dataframe_from_db("SELECT * FROM gold.customer_dim where customer_type ='registered';")

    customers_to_be_inserted = known_customers.merge(all_existing_customers , how='left',on=['customer_id'],indicator=True ,suffixes=["" , "X"])
    customers_to_be_inserted = customers_to_be_inserted[customers_to_be_inserted['_merge'] == 'left_only']
    customers_to_be_inserted = customers_to_be_inserted[ ['customer_id' , 'customer_type']]
    customers_to_be_inserted = customers_to_be_inserted.drop_duplicates(subset=['customer_id' ,'customer_type'])
    
    customers_to_be_inserted = pd.concat([customers_to_be_inserted , unknown_customers],ignore_index=True)
    customers_to_be_inserted = customers_to_be_inserted [ ['customer_id' , 'customer_type']]
    # 0 if len(customers_to_be_inserted) == 0  else db_conn.load_dataframe_into_db(customers_to_be_inserted , 'gold','customer_dim')
    return customers_to_be_inserted


def fill_product_dim_table(cleansed_data:pd.DataFrame ,db_conn:DatabaseConnection)-> pd.DataFrame:
    product_dim = cleansed_data[ ['description' , 'stock_code']]
    # 0 if len(product_dim) == 0 else db_conn.load_dataframe_into_db(product_dim , 'gold','product_dim')
    return product_dim


def fill_country_dim_table(cleansed_data: pd.DataFrame , db_conn: DatabaseConnection)-> pd.DataFrame:
    
    new_country_batch = cleansed_data[ ['country']]

    all_countries = db_conn.read_dataframe_from_db("SELECT * FROM gold.country_dim;")

    new_batch_to_be_inserted = new_country_batch.merge(all_countries , how = 'left' , on=['country'],indicator=True)
    
    new_batch_to_be_inserted = new_batch_to_be_inserted[ new_batch_to_be_inserted['_merge']=='left_only']
    new_batch_to_be_inserted = new_batch_to_be_inserted[['country']].drop_duplicates(subset=['country'])

    # 0 if len(new_batch_to_be_inserted) == 0 else db_conn.load_dataframe_into_db(new_batch_to_be_inserted ,'gold' , 'country_dim')
    return new_batch_to_be_inserted

def fill_sales_fact_table(db_conn:DatabaseConnection , cleansed_data:pd.DataFrame )-> pd.DataFrame:

    # Load Dims

    customer_dim = db_conn.load_dataframe_into_db("SELECT FROM * GOLD.CUSTOMER_DIM;")
    country_dim = db_conn.load_dataframe_into_db("SELECT FROM * GOLD.COUNTRY_DIM;")


    cleansed_data['date_key'] = cleansed_data['invoice_date'].apply(lambda x: int(x.strftime("%Y%m%d")))
    cleansed_data['customer_id'] = cleansed_data['customer_id'].apply(lambda x: '-1' if x == np.nan else x)    
    cleansed_data = cleansed_data.merge(country_dim , on=['country'] , how='left')
    cleansed_data = cleansed_data.merge(customer_dim ,how='left',on=['customer_id'],suffixes=["", "X"])

    sales_fact_dim = cleansed_data[ ['silver_sales_id', 'date_key' , 'product_key' , 'country_key' , 'customer_key' , 'invoice_number',
                                     'quantity' , 'unit_price', 'is_return' , 'total_line'] ]




    return sales_fact_dim
    
def load_dim_tables_into_gold_schema(db_conn:DatabaseConnection , dim_tables:list):


    for table in dim_tables:
        db_conn.load_dataframe_into_db(table["table_data"] , 'gold',table['table_name'])

def run_load(db_conn:DatabaseConnection):

    cleansed_data = read_data_from_silver(db_conn)
    date_dim = fill_date_dim_table(db_conn)
    customer_dim = fill_customer_dim_table(db_conn)
    product_dim = fill_product_dim_table(db_conn)
    country_dim = fill_country_dim_table(db_conn)

    dim_metadata = [ 
        {
            "table_name":"date_dim",
            "table_data":date_dim
        },
        {
            "table_name":"customer_dim",
            "table_data":customer_dim
        },
        {
            "table_name":"product_dim",
            "table_data":product_dim
        },
        {
            "table_name":"country_dim",
            "table_data":country_dim
        }
     
    ]

    load_dim_tables_into_gold_schema(db_conn , dim_metadata)

    sales_fact =  fill_sales_fact_table(cleansed_data)

    total_inserted_into_gold = db_conn.load_dataframe_into_db(sales_fact , 'gold','sales_fact')

    silver_data_len = len(cleansed_data)

    return {
        "total_from_silver": silver_data_len,
        "total_inserted_into_silver" : total_inserted_into_gold,
        "status" : "Success" if (silver_data_len == total_inserted_into_gold) else "Failed"
    }







In [15]:
db_conn = DatabaseConnection()
cleansed = read_data_from_silver(db_conn)
product_dim = fill_product_dim_table(cleansed,db_conn)
customer_dim = fill_customer_dim_table(cleansed[['customer_id' , 'customer_type']],db_conn)
date_dim = fill_date_dim_table(cleansed , db_conn)
country_dim = fill_country_dim_table(cleansed , db_conn)



   date_key   full_date  day_of_week  day_of_month   day_name  week_of_year  \
0  20101201  2010-12-01            3             1  Wednesday            48   
1  20101201  2010-12-01            3             1  Wednesday            48   
2  20101201  2010-12-01            3             1  Wednesday            48   
3  20101201  2010-12-01            3             1  Wednesday            48   
4  20101201  2010-12-01            3             1  Wednesday            48   

   month month_name  quarter  year  is_weekend  
0     12   December        4  2010       False  
1     12   December        4  2010       False  
2     12   December        4  2010       False  
3     12   December        4  2010       False  
4     12   December        4  2010       False  
10051
final = 15


In [ ]:
cleansed = read_data_from_silver(db_conn)
country_dim = fill_country_dim_table(cleansed , db_conn)


In [ ]:
db_conn = DatabaseConnection()
cleansed = read_data_from_silver(db_conn)
cust = fill_customer_dim_table(cleansed , db_conn)

In [ ]:


country_dim = fill_country_dim_table(cleansed , db_conn)


In [ ]:
country_dim

In [ ]:
tot = db_conn.read_dataframe_from_db("select * from gold.country_dim;")
tot

In [ ]:


df = pd.DataFrame({'country' : ['']})

df = df.merge(tot , on=['country'] , how='left',indicator=True)
df

In [8]:
db_conn = DatabaseConnection()
cleansed = read_data_from_silver(db_conn)
cleansed.head()

,silver_sales_id,raw_sales_id,invoice_number,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,ingestion_date,customer_type,is_return,total_line,transformed_date,product_key
0,62479,500997,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2026-01-04 21:49:26.261442,registered,False,15.30,2026-01-06 20:19:46.629438,47edf121-eb2c-11f0-acb4-000c2943d053
1,62480,500998,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2026-01-04 21:49:26.261442,registered,False,20.34,2026-01-06 20:19:46.629438,47ef0bee-eb2c-11f0-95d1-000c2943d053
2,62481,500999,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2026-01-04 21:49:26.261442,registered,False,22.00,2026-01-06 20:19:46.629438,47ef0cb1-eb2c-11f0-8074-000c2943d053
3,62482,501000,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2026-01-04 21:49:26.261442,registered,False,20.34,2026-01-06 20:19:46.629438,47ef0d0b-eb2c-11f0-b108-000c2943d053
4,62483,501001,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2026-01-04 21:49:26.261442,registered,False,20.34,2026-01-06 20:19:46.629438,47ef0d4d-eb2c-11f0-9582-000c2943d053


In [11]:
dd = fill_customer_dim_table(cleansed , db_conn )

In [12]:
dd

2569

In [27]:
df = fill_sales_fact_table(cleansed , db_conn.read_dataframe_from_db("SELECT * FROM GOLD.customer_dim;") ,
                            db_conn.read_dataframe_from_db("SELECT * FROM gold.product_dim;"),
                            db_conn.read_dataframe_from_db("SELECT * FROM gold.date_dim;") , 
                            db_conn.read_dataframe_from_db("SELECT * FROM gold.country_dim;"))
df.head(2)

,silver_sales_id,date_key,product_key,country_key,customer_key,invoice_number,quantity,unit_price,is_return,total_line
0,72530,20101201,7aae12a0-ebb4-11f0-8370-000c2943d053,46,3543,536365,6,2.55,False,15.30
1,72531,20101201,7aaffed6-ebb4-11f0-aee3-000c2943d053,46,3543,536365,6,3.39,False,20.34


In [21]:
country_dim

15

In [ ]:
import pandas as pd
import sys
import os
# Add parent directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.db_connection import DatabaseConnection
from datetime import datetime
from sqlalchemy import text
import numpy as np

def get_max_transformed_date(db_conn: DatabaseConnection) -> datetime.date:
    """
        Function to return the max date of the column from the bronze.raw_sales
        to be used in fetching the new inserted data into bronze
    """

    with db_conn.engine.begin() as query_conn:
        result = query_conn.execute(text(""" SELECT max(transformed_date) FROM SILVER.cleansed_sales; """))
    return result.scalar().date()



def read_data_from_silver(db_conn: DatabaseConnection) ->pd.DataFrame:

    return db_conn.read_dataframe_from_db(f"SELECT * FROM Silver.cleansed_sales where transformed_date >= DATE '{get_max_transformed_date(db_conn)}'")



def fill_date_dim_table(cleansed_data: pd.DataFrame, db_connection: DatabaseConnection) -> pd.DataFrame:
    """
    Fill date dimension table for the ingestion date.

    Returns:
        DataFrame with single date dimension record
    """
    

    date_dim = cleansed_data[['invoice_date']].copy()

    date_dim['date_key'] = date_dim['invoice_date'].apply(lambda x: int(x.strftime("%Y%m%d")))
    date_dim['full_date'] = date_dim['invoice_date'].apply(lambda x: x.date())
    date_dim['day_of_week'] = date_dim['invoice_date'].apply(lambda x: x.date().weekday()+ 1)
    date_dim['day_of_month'] = date_dim['invoice_date'].apply(lambda x: x.date().day)
    date_dim['day_name'] = date_dim ['invoice_date'].apply(lambda x: x.date().strftime("%A"))
    date_dim['week_of_year'] = date_dim['invoice_date'].apply(lambda x: x.date().isocalendar()[1])
    date_dim['month'] = date_dim['invoice_date'].apply(lambda x: x.date().month)
    date_dim['month_name']=date_dim['invoice_date'].apply(lambda x: x.date().strftime("%B"))
    date_dim['quarter']=date_dim['invoice_date'].apply(lambda x: (x.date().month - 1 )// 3 + 1)
    date_dim['year'] = date_dim['invoice_date'].apply(lambda x: x.date().year)
    date_dim['is_weekend'] = date_dim['invoice_date'].apply(lambda x: x.date().weekday() >= 5)


    all_stored_dates = db_connection.read_dataframe_from_db("SELECT * FROM GOLD.DATE_DIM;")

    date_dim = date_dim.merge(all_stored_dates , how='left' , on=['date_key'] , indicator=True, suffixes=("","X"))


    date_dim = date_dim[date_dim['_merge'] == 'left_only']
    

    date_dim = date_dim[["date_key","full_date","day_of_week","day_of_month","day_name","week_of_year","month","month_name","quarter","year","is_weekend"]]
    print(date_dim.head())
    date_dim = date_dim.drop_duplicates(subset=['date_key'])
    # 0 if len(date_dim) == 0 else db_connection.load_dataframe_into_db(date_dim , 'gold' , 'date_dim')    
    return  date_dim



def fill_customer_dim_table(cleansed_data: pd.DataFrame , db_conn:DatabaseConnection) -> pd.DataFrame:

    customer_data = cleansed_data.copy()
    
    known_customers = customer_data[ customer_data['customer_type'] =='registered']
    unknown_customers = customer_data[ customer_data['customer_type'] =='guest']
    
    all_existing_customers = db_conn.read_dataframe_from_db("SELECT * FROM gold.customer_dim where customer_type ='registered';")

    customers_to_be_inserted = known_customers.merge(all_existing_customers , how='left',on=['customer_id'],indicator=True ,suffixes=["" , "X"])
    customers_to_be_inserted = customers_to_be_inserted[customers_to_be_inserted['_merge'] == 'left_only']
    customers_to_be_inserted = customers_to_be_inserted[ ['customer_id' , 'customer_type']]
    customers_to_be_inserted = customers_to_be_inserted.drop_duplicates(subset=['customer_id' ,'customer_type'])
    
    customers_to_be_inserted = pd.concat([customers_to_be_inserted , unknown_customers],ignore_index=True)
    customers_to_be_inserted = customers_to_be_inserted [ ['customer_id' , 'customer_type']]
    # 0 if len(customers_to_be_inserted) == 0  else db_conn.load_dataframe_into_db(customers_to_be_inserted , 'gold','customer_dim')
    return customers_to_be_inserted


def fill_product_dim_table(cleansed_data:pd.DataFrame ,db_conn:DatabaseConnection)-> pd.DataFrame:
    product_dim = cleansed_data[ ['description' , 'stock_code']]
    # 0 if len(product_dim) == 0 else db_conn.load_dataframe_into_db(product_dim , 'gold','product_dim')
    return product_dim


def fill_country_dim_table(cleansed_data: pd.DataFrame , db_conn: DatabaseConnection)-> pd.DataFrame:
    
    new_country_batch = cleansed_data[ ['country']]

    all_countries = db_conn.read_dataframe_from_db("SELECT * FROM gold.country_dim;")

    new_batch_to_be_inserted = new_country_batch.merge(all_countries , how = 'left' , on=['country'],indicator=True)
    
    new_batch_to_be_inserted = new_batch_to_be_inserted[ new_batch_to_be_inserted['_merge']=='left_only']
    new_batch_to_be_inserted = new_batch_to_be_inserted[['country']].drop_duplicates(subset=['country'])

    # 0 if len(new_batch_to_be_inserted) == 0 else db_conn.load_dataframe_into_db(new_batch_to_be_inserted ,'gold' , 'country_dim')
    return new_batch_to_be_inserted

def fill_sales_fact_table(db_conn:DatabaseConnection , cleansed_data:pd.DataFrame )-> pd.DataFrame:

    # Load Dims

    customer_dim = db_conn.read_dataframe_from_db("SELECT * FROM  GOLD.CUSTOMER_DIM;")
    country_dim = db_conn.read_dataframe_from_db("SELECT  * FROM  GOLD.COUNTRY_DIM;")


    cleansed_data['date_key'] = cleansed_data['invoice_date'].apply(lambda x: int(x.strftime("%Y%m%d")))
    cleansed_data['customer_id'] = cleansed_data['customer_id'].apply(lambda x: '-1' if x == np.nan else x)    
    cleansed_data = cleansed_data.merge(country_dim , on=['country'] , how='left')
    cleansed_data = cleansed_data.merge(customer_dim ,how='left',on=['customer_id'],suffixes=["", "X"])

    sales_fact_dim = cleansed_data[ ['silver_sales_id', 'date_key' , 'product_key' , 'country_key' , 'customer_key' , 'invoice_number',
                                     'quantity' , 'unit_price', 'is_return' , 'total_line'] ]




    return sales_fact_dim
    
def load_dim_tables_into_gold_schema(db_conn:DatabaseConnection , dim_tables:list):


    for table in dim_tables:
        if len(table['table_data']):
            db_conn.load_dataframe_into_db(table["table_data"] , 'gold',table['table_name'])

def run_load(db_conn:DatabaseConnection):

    cleansed_data = read_data_from_silver(db_conn)
    date_dim = fill_date_dim_table(cleansed_data[['invoice_date']],db_conn)
    customer_dim = fill_customer_dim_table(cleansed_data[['customer_id','customer_type']],db_conn)
    product_dim = fill_product_dim_table(cleansed_data[['stock_code' , 'description']] ,db_conn)
    country_dim = fill_country_dim_table(cleansed_data[['country']],db_conn``)

    dim_metadata = [ 
        {
            "table_name":"date_dim",
            "table_data":date_dim
        },
        {
            "table_name":"customer_dim",
            "table_data":customer_dim
        },
        {
            "table_name":"product_dim",
            "table_data":product_dim
        },
        {
            "table_name":"country_dim",
            "table_data":country_dim
        }
     
    ]

    load_dim_tables_into_gold_schema(db_conn , dim_metadata)

    sales_fact =  fill_sales_fact_table(db_conn , cleansed_data)

    total_inserted_into_gold = db_conn.load_dataframe_into_db(sales_fact , 'gold','sales_fact')

    silver_data_len = len(cleansed_data)

    return {
        "total_from_silver": silver_data_len,
        "total_inserted_into_silver" : total_inserted_into_gold,
        "status" : "Success" if (silver_data_len == total_inserted_into_gold) else "Failed"
    }



print(run_load(DatabaseConnection()))




Empty DataFrame
Columns: [date_key, full_date, day_of_week, day_of_month, day_name, week_of_year, month, month_name, quarter, year, is_weekend]
Index: []


KeyboardInterrupt: 

In [ ]:
print("X")